In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent.notebook_imports import *

In [4]:
from functools import cache

In [5]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from waymo_agent.action_heuristic.heuristic_simple import PricingAgent, DispatchAgent, RepositionAgent
from waymo_agent.models.ppo_model import RideShareActorCritic

In [6]:
from kret_sandbox.VIS import dtt
from kret_sandbox.exp_decay import exp_decay_half_life, get_gamma_from_half_life

In [7]:
def get_obvs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides

# Run Sim

In [8]:
config = EnvConfig(max_episode_steps=60 * 3)
plt_cfg = PlotConfig()

In [9]:
GAMMA = get_gamma_from_half_life(config.max_episode_steps // 2)
round(GAMMA, 5)

0.99233

In [10]:
def discounted_rewards(rewards: np.ndarray, gamma: float = 0.997):
    """
    Discount rewards using discount factor gamma.
    """
    disc_schedule = exp_decay_half_life(len(rewards), gamma=gamma)
    return rewards * disc_schedule


def get_act_dict(agents: tuple[PricingAgent, DispatchAgent, RepositionAgent], obs: ObservationDict) -> ActionDict:
    price_agent, dispatch_agent, reposition_agent = agents
    prices = price_agent.price(obs)
    dispatch_actions = dispatch_agent.dispatch(obs)
    reposition_actions = reposition_agent.reposition(obs)

    action_agent: ActionDict = {
        "prices": prices,
        "dispatch": dispatch_actions,
        "reposition": reposition_actions,
    }
    return action_agent

In [11]:
def run_heuristic_simulation(env_curr: RideShareEnv, num_iter: int = 1):
    REWARDS: list[list[float]] = []
    OBS: list[list[ObservationDict]] = []
    ACT: list[list[ActionDict]] = []

    agents = (PricingAgent(env_curr.config), DispatchAgent(env_curr), RepositionAgent(env_curr))

    tqdm.write("Starting heuristic simulation...")

    for iter_count in tqdm(range(num_iter)):
        rews_raw: list[float] = []
        obs_list: list[ObservationDict] = []
        act_list: list[ActionDict] = []
        done = False
        obs, info = env_curr.reset()

        while not done:
            obs: ObservationDict
            obs_list.append(obs)

            action_agent: ActionDict = get_act_dict(agents, obs)
            act_list.append(action_agent)

            obs, reward, term, done, info = env_curr.step(action_agent)  # type: ignore

            rews_raw.append(reward)
            if term:
                tqdm.write(f"WARNING: Episode terminated at step {env_curr.current_step}")
            done = done or term

        rews = discounted_rewards(np.array(rews_raw), gamma=GAMMA).tolist()
        REWARDS.append(rews)
        OBS.append(obs_list)
        ACT.append(act_list)
    return (REWARDS), OBS, ACT, env_curr, agents

In [12]:
env_heuristic = RideShareEnv(config, plt_cfg)
G = env_heuristic.G
cfg = env_heuristic.config

Assigned lambda values to nodes. Total lambda: 2.2383 (target: 2.3460)


/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [13]:
REWARDS, OBS, ACT, env_curr, agents = run_heuristic_simulation(env_heuristic, num_iter=5)

Starting heuristic simulation...


  0%|          | 0/5 [00:00<?, ?it/s]


KeyError: "['cust_id'] not in index"

In [ ]:
arr = np.array(REWARDS)
arr.shape

(5, 181)

In [ ]:
arr2 = arr.reshape(-1, arr.shape[0])
arr2.shape

(181, 5)

In [ ]:
pd.DataFrame(arr2)

,0,1,2,3,4
0,0.000000,0.000000,1.199640,0.135630,0.379301
1,0.233252,-0.157225,-0.122982,-0.591333,-0.706863
2,0.234375,0.420901,0.985467,-0.144414,2.896067
3,-0.217176,1.297572,0.865732,1.496606,0.735964
4,0.182475,1.785050,0.541758,0.998995,2.407283
...,...,...,...,...,...
176,0.475577,3.646878,2.599660,6.503858,2.495791
177,0.694825,0.727968,2.791800,3.835558,1.381809
178,2.385986,4.795629,0.631846,0.620727,0.643207
179,3.284221,0.377831,0.750801,0.018138,2.549395


In [ ]:
from waymo_agent.graph_env.ENV import RideShareEnv
from waymo_agent.models.ppo_model import (
    train_ppo,
    PPOTrainConfig,
    RideShareActorCritic,
    obs_numpy_to_torch,
    action_torch_to_numpy,
)

In [ ]:
env_model = RideShareEnv(config, plt_cfg)
model = RideShareActorCritic(env_model)

Assigned lambda values to nodes. Total lambda: 2.2776 (target: 2.3460)


/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [ ]:
obs, info = env_model.reset()

In [ ]:
def obs_numpy_to_torch(obs: dict[str, np.ndarray] | ObservationDict) -> dict[str, torch.Tensor]:
    """Convert env obs (numpy) -> torch tensors on DEVICE."""
    ret = {}

    for k, v in obs.items():
        if isinstance(v, EnrichedDF):
            v = v.to_obs_numpy()
        elif isinstance(v, pd.DataFrame):
            v = v.to_numpy()

        ret[k] = torch.as_tensor(v, device=DEVICE, dtype=torch.float32)

    return ret

In [ ]:
@torch.no_grad()
def run_model_simulation(
    env_curr: RideShareEnv,
    model: RideShareActorCritic,
    num_iter: int = 1,
    gamma: float = GAMMA,
    deterministic: bool = False,
) -> tuple[list[list[float]], list[list[dict[str, np.ndarray]]], list[list[dict[str, np.ndarray]]]]:
    """
    Returns:
      REWARDS: list[episode][t] discounted reward_t
      OBS:     list[episode][t] obs dict (numpy)
      ACT:     list[episode][t] action dict (numpy)
    """
    model.eval()

    REWARDS: list[list[float]] = []
    OBS: list[list[dict[str, np.ndarray]]] = []
    ACT: list[list[dict[str, np.ndarray]]] = []

    for _ in tqdm(range(num_iter), desc="model rollout"):
        obs_np, _info = env_curr.reset()
        done = False

        rews_raw: list[float] = []
        obs_list: list[dict[str, np.ndarray]] = []
        act_list: list[dict[str, np.ndarray]] = []

        while not done:
            obs_list.append(obs_np)

            obs_t = obs_numpy_to_torch(obs_np)
            act_t = model.act(obs_t, deterministic=deterministic)
            act_np = action_torch_to_numpy(act_t)
            act_list.append(act_np)

            obs_np, reward, terminated, truncated, _info = env_curr.step(act_np)  # type: ignore[arg-type]
            rews_raw.append(float(reward))
            done = bool(terminated) or bool(truncated)

        rews = discounted_rewards(np.array(rews_raw, dtype=np.float32), gamma=gamma).tolist()
        REWARDS.append(t.cast(list[float], rews))
        OBS.append(obs_list)
        ACT.append(act_list)

    return REWARDS, OBS, ACT

In [ ]:
for obsname, value in obs.items():
    if isinstance(value, np.ndarray):
        print(f"{obsname}: ndarray, shape={value.shape}, dtype={value.dtype}")
    elif isinstance(value, pd.DataFrame):
        print(f"{obsname}: DataFrame, shape={value.shape}, dtypes={value.dtypes}")

globals: ndarray, shape=(5,), dtype=float64
supply_demand_ratio: ndarray, shape=(3,), dtype=float64
vehicles: DataFrame, shape=(24, 4), dtypes=loc_x_norm    float64
loc_y_norm    float64
battery       float64
status           int8
dtype: object
pending_requests: DataFrame, shape=(50, 9), dtypes=pickup_x_norm              float64
pickup_y_norm              float64
dropoff_x_norm             float64
dropoff_y_norm             float64
distance_meters            float64
est_cost                   float64
max_wait_time      timedelta64[ns]
wait_time          timedelta64[ns]
status                       int64
dtype: object
active_rides: DataFrame, shape=(24, 9), dtypes=pickup_x_norm                       float32
pickup_y_norm                       float32
dropoff_x_norm                      float32
dropoff_y_norm                      float32
price                               float32
est_cost                            float32
total_trip_distance_meters          float32
trip_distance_remain

In [ ]:
run_model_simulation(env_model, model, num_iter=1)

model rollout:   0%|          | 0/1 [00:00<?, ?it/s]

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [ ]:
obs["pending_requests"]

waymo_agent.data_classes.requests.RequestDF

In [ ]:
env = RideShareEnv()
cfg = PPOTrainConfig(total_steps=5, rollout_len=3)
model, logs = train_ppo(env, cfg=cfg)

In [ ]:
# env.breadcrumbs["rewards"].cumsum().plot(title="Cumulative Reward over Time")
# plt.xlabel("Time Step")
# plt.ylabel("Cumulative Reward")
# plt.show()

In [ ]:
# fig, ax = env.render()
# fig